# Week 44, Day 6: Forge: Week 44 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
V = 0
for _ in range(3):
    V += 0.25 * (1 - V)
print("1.", V)
print("2. delivered", 1 - 0.6, " omitted", -0.6)
print("3.", round(0.5 * 0.2 / (0.5 * 0.2 + 0.5 * 0.1), 3))

### Exercise: Part A: a sticky participant

Write nll(par, ch, rw, sticky): utilities are beta Q plus kappa for the previously chosen option (kappa = 0 when sticky is False). Fit both models with L-BFGS-B and compute AIC and BIC.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def simulate(alpha, beta, kappa, T=400, seed=1):
    g = np.random.default_rng(seed); p = np.array([0.7, 0.3]); Q = np.zeros(2); prev = -1; ch, rw = [], []
    for t in range(T):
        if t % 100 == 0 and t > 0:
            p = p[::-1]
        stick = np.zeros(2)
        if prev >= 0:
            stick[prev] = 1
        u = beta * Q + kappa * stick
        a = int(g.random() < 1 / (1 + np.exp(-(u[1] - u[0])))); rr = float(g.random() < p[a])
        Q[a] += alpha * (rr - Q[a]); prev = a; ch.append(a); rw.append(rr)
    return np.array(ch), np.array(rw)

def nll(par, ch, rw, sticky=True):
    alpha, beta = par[0], par[1]; kappa = par[2] if sticky else 0.0
    Q = np.zeros(2); prev = -1; L = 0.0
    for a, rr in zip(ch, rw):
        ...
    return L

ch, rw = simulate(0.3, 4.0, 1.5)
n = len(ch)
fits = {"basic": minimize(nll, [0.5, 3], args=(ch, rw, False), method="L-BFGS-B", bounds=[(0.01, 0.99), (0.1, 20)]),
        "sticky": minimize(nll, [0.5, 3, 0], args=(ch, rw, True), method="L-BFGS-B", bounds=[(0.01, 0.99), (0.1, 20), (-5, 5)])}
k = {"basic": 2, "sticky": 3}
aic = {m: ... for m in fits}
bic = {m: ... for m in fits}
for m in fits:
    print(f"{m:6s} params {np.round(fits[m].x, 2)}, NLL {fits[m].fun:.1f}, AIC {aic[m]:.1f}, BIC {bic[m]:.1f}")

In [ ]:
import numpy as np
def test_compare():
    "both AIC and BIC prefer the sticky model"
    assert aic["sticky"] < aic["basic"] and bic["sticky"] < bic["basic"]
def test_recover():
    "the sticky model recovers kappa within 0.5 and beta within 1"
    assert abs(fits["sticky"].x[2] - 1.5) < 0.5 and abs(fits["sticky"].x[1] - 4.0) < 1.0
def test_bias():
    "the basic model overestimates beta, absorbing the stickiness"
    assert fits["basic"].x[1] > fits["sticky"].x[1] + 1

test_compare()
test_recover()
test_bias()
print("All checks passed")

### Exercise: Part B: imputing the distribution

Write loss(z) as above, then run scipy.optimize.minimize from five random starting sets of 50 samples (uniform between the smallest and largest expectile) and keep the best.

In [ ]:
g = np.random.default_rng(7)
xs = np.where(g.random(200000) < 0.3, g.normal(1, 0.3, 200000), g.normal(6, 0.8, 200000))
taus = np.linspace(0.05, 0.95, 19)

def expectiles(samples):
    out = []
    for tau in taus:
        lo, hi = samples.min(), samples.max()
        for _ in range(40):
            m = (lo + hi) / 2
            f = tau * np.mean(np.maximum(samples - m, 0)) - (1 - tau) * np.mean(np.maximum(m - samples, 0))
            lo, hi = (m, hi) if f > 0 else (lo, m)
        out.append(m)
    return np.array(out)

E = expectiles(xs)                       # the "reversal points" of nineteen neurons

def loss(z):
    ...

best = None
for k_ in range(5):
    res = minimize(loss, g.uniform(E.min(), E.max(), 50), method="L-BFGS-B")
    if best is None or res.fun < best.fun:
        best = res
z = best.x
print(f"loss {best.fun:.2e}; fraction of decoded samples below 3.5: {np.mean(z < 3.5):.2f}; decoded mean {z.mean():.2f} (true {xs.mean():.2f})")
plt.hist(xs, 80, density=True, alpha=0.4, label="true"); plt.hist(z, 20, density=True, alpha=0.6, label="decoded"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the best imputation matches the expectiles almost exactly"
    assert best.fun < 1e-4
def test_shape():
    "the decoded distribution puts about 30% of its mass in the low mode and has the right mean"
    assert abs(np.mean(z < 3.5) - 0.3) < 0.1 and abs(z.mean() - E[9]) < 0.15

test_fit()
test_shape()
print("All checks passed")